## Databricks Agent Multi-Tool Setup

### Installing Utilities and Libraries

In [ ]:
%pip install \
    databricks-sdk==0.49.0 \
    openai-agents==0.22.0 \
    mcp==2.0.0 \
    databricks-mcp==0.9.2 \
    "mlflow>=3.1"

### Restart your Python Environment

In [ ]:
dbutils.library.restartPython()

### Setup your Environment

In [ ]:
from databricks.sdk import WorkspaceClient

# Get Databricks runtime authentication
w = WorkspaceClient()

headers = w.config.authenticate()
token = headers["Authorization"].replace("Bearer ", "")
workspace_host = w.config.host.rstrip("/")

### Define your MCP Server URLs

In [ ]:
from databricks_mcp import DatabricksMCPClient
from databricks.sdk import WorkspaceClient

custom_mcp_server_url = "YOUR-CUSTOM-MCP-SERVER-URL-GOES-HERE"

code_interpreter_mcp_server_url = (
    f"{workspace_host}/api/2.0/mcp/functions/"
    f"system/ai/python_exec"
)

### Create and Execute the Agent

In [ ]:
from agents import (
    Agent,
    Runner,
    AsyncOpenAI,
    OpenAIChatCompletionsModel,
    set_tracing_disabled
)
from agents.mcp import MCPServerStreamableHttp

# Create an OpenAI-compatible client for Databricks
client = AsyncOpenAI(
    api_key=token,
    base_url=f"{workspace_host}/serving-endpoints"
)

# Configure the Databricks model
model = OpenAIChatCompletionsModel(
    model="databricks-claude-sonnet-4-5",
    openai_client=client
)

async with (
    MCPServerStreamableHttp(
        name="MSLearn-MCP-Server",
        params={
            "url": custom_mcp_server_url,
            "headers": {
                "Authorization": f"Bearer {token}"
            }
        }
    ) as custom_mcp_server,

    MCPServerStreamableHttp(
        name="code-interpreter",
        params={
            "url": code_interpreter_mcp_server_url,
            "headers": {
                "Authorization": f"Bearer {token}"
            }
        }
    ) as code_interpreter
):

    # Don't send Agents SDK traces to OpenAI
    set_tracing_disabled(True)
    
    # Create agent
    agent = Agent(
        name="Multi-Tool-Agent",
        instructions=(
            "You are a helpful AI assistant. "
            "Use the Python code interpreter for calculations "
            "and code execution. Use Microsoft Learn "
            "for Microsoft technical documentation and "
            "learning resources."
        ),
        model=model,
        mcp_servers=[code_interpreter, custom_mcp_server]
    )

    # Run agent
    result = await Runner.run(
        agent,
        "Write code and execute to calculate standard deviation"
        "of all natural numbers upto 10"
    )

    print(result.final_output)